In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

# 1. Image Preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), # Crucial for medical images (feet can be at any angle)
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # ImageNet defaults
])

# 2. Load the Patches
# Replace 'path_to_patches' with your actual folder path
# Use the 'r' prefix to handle Windows backslashes correctly
dataset = datasets.ImageFolder(root=r"C:\Users\pc\Downloads\DFU\Patches", transform=transform)

# Split into Train and Validation (80/20 split)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=16, shuffle=False)

# Load pre-trained EfficientNet
model = models.efficientnet_b0(weights='DEFAULT')

# Freeze the early layers (optional, but helps speed up initial training)
for param in model.features.parameters():
    param.requires_grad = False

# Replace the final classifier layer for 2 classes
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 2) # Binary output
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

import torch.optim as optim

# Loss function: CrossEntropy is standard for classification
criterion = nn.CrossEntropyLoss()

# Optimizer: Adam is fast and reliable for hackathons
# We only optimize parameters that are NOT frozen
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

num_epochs = 10 # Start with 10 to see how it performs
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # 1. Reset gradients
        optimizer.zero_grad()

        # 2. Forward pass (Make a guess)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # 3. Backward pass (Learn)
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.2f}%")

print("Training Complete!")
torch.save(model.state_dict(), 'dfu_model_v1.pth')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\pc/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100.0%


Epoch [1/10] - Loss: 0.1590 - Acc: 93.96%
Epoch [2/10] - Loss: 0.0934 - Acc: 97.04%
Epoch [3/10] - Loss: 0.0886 - Acc: 96.80%
Epoch [4/10] - Loss: 0.0753 - Acc: 97.27%
Epoch [5/10] - Loss: 0.1080 - Acc: 96.68%
Epoch [6/10] - Loss: 0.0829 - Acc: 96.56%
Epoch [7/10] - Loss: 0.0915 - Acc: 96.56%
Epoch [8/10] - Loss: 0.0718 - Acc: 97.75%
Epoch [9/10] - Loss: 0.0524 - Acc: 98.10%
Epoch [10/10] - Loss: 0.0914 - Acc: 96.92%
Training Complete!
